<a href="https://colab.research.google.com/github/Aryan2079/LLM_finetuning/blob/main/instruction_finetuning_on_domain_specific_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
dataset_name = "Amod/mental_health_counseling_conversations"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token=tokenizer.eos_token

In [ ]:
non_instructional_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
prompt = "Clinical trails demonstrated that combining Atorvastatin with Ezetimibe"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = non_instructional_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trails demonstrated that combining Atorvastatin with Ezetimibe has a significant reduction in serum LDL cholesterol, triglycerides and the risk of heart disease.
Our goal is to provide safe and effective nutritional supplements with the highest quality standards possible to help people manage their health.


**Now lets start instruction finetuning**

In [ ]:
dataset = load_dataset(dataset_name, split="train")

In [ ]:
dataset[0]

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someh

In [ ]:
format_template = """[INST] {} [/INST] {}""" # this INST thing is just a format thing. u can choose other formats too.

In [ ]:
def format_row(example):
  question = example["Context"]
  answer = example["Response"]
  example["text"] = format_template.format(question, answer)
  return example

In [ ]:
formatted_dataset = dataset.map(format_row)

In [ ]:
formatted_dataset

Dataset({
    features: ['Context', 'Response', 'text'],
    num_rows: 3512
})

In [ ]:
formatted_dataset[0]

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someh

**Trying with custom dataset**

In [ ]:
dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [ ]:
def format_example(example):
  prompt = f"### Instruction:\n{example['instruction']}\n{example['input']}\n### Response:\n{example['output']}"
  return {"text": prompt}

In [ ]:
dataset = dataset.map(format_example)
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 5
})

In [ ]:
dataset[0]

{'instruction': 'Explain the mechanism of action of Metformin.',
 'input': None,
 'output': 'Metformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.',
 'text': '### Instruction:\nExplain the mechanism of action of Metformin.\nNone\n### Response:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'}

In [ ]:
def tokenize_fn(example):
  tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True)

In [ ]:
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    bias="none"
)

In [ ]:
model = get_peft_model(non_instructional_model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir = "./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=12.37717310587565, metrics={'train_runtime': 7.8217, 'train_samples_per_second': 1.918, 'train_steps_per_second': 0.384, 'total_flos': 47722235166720.0, 'train_loss': 12.37717310587565, 'epoch': 3.0})

In [ ]:
model_path = "/content/tinyllama-instruction/checkpoint-3"


In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = "Explain the mechanism of action of Metformin."

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model output:

Explain the mechanism of action of Metformin. 24.5 Explain the mechanism of action of Metformin.
It's not so much about "what" as it is about "how" – how the body uses the drug. You can look at Metformin on its own, but you have to understand the whole picture to get a true understanding of its effectiveness.
The mechanism of action of metformin was first described in 1970 by researchers at Johns Hopkins University and University


**In the above example, we masked the entire text ie. we use input_ids and labels for the entire text. but we can also only mask the reponse of the model since that is what matters.**

In [ ]:
# tokenization with response masking
def tokenize_and_mask(example):
  text = example["text"]

  #tokenize full text
  enc = tokenizer(text, truncation=True, padding="max_length",max_length=512)
  input_ids = enc["input_ids"]

  #find where '### Response:' starts
  response_marker = "### Response:"
  response_start = text.find(response_marker)

  if response_start != -1:
    #Token index where response begins
    response_token_start = len(tokenizer(text[:response_start])["input_ids"])
  else:
    response_token_start = 0 # if the marker is not found

  # clone labels and mask out everything before "Response"
  labels = input_ids.copy()
  labels[:response_token_start] = [-100] * response_token_start

  enc["labels"] = labels
  return enc

# Apply tokenization
tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")

# LoRA config
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    bias="none"
)

# load base model
non_instructional_trained_model = AutoModelForCausalLM.from_pretrained(
    "/content/tinyllama-instruction/checkpoint-3",
    torch_dtype=torch.float16,
    device_map="auto"
)

model = get_peft_model(non_instructional_trained_model, lora_config)

# Training setup
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

trained = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized
)

trainer.train()

trainer.save_model("/content/tinyllama-instruction")
tokenizer.save_pretrained("/content/tinyllama-instruction")

trained_model = AutoModelForCausalLM.from_pretrained("/content/tinyllama-instruction", device_map="cuda")
# trained_model = PeftModel.from_pretrained(base_model, "/content/tinyllama-instruction")

prompt = "### Instructio:\nwhat is Ezetimibe?\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Tokenization + masking done.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 126.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 21.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.22 GiB is allocated by PyTorch, and 190.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)